In [75]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

In [76]:
import numpy as np
from numpy import pi
import random
from functools import partial
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import eigh
from scipy.special import erf
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.classical import expr
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator, random_statevector
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate, CPhaseGate, UnitaryGate, U3Gate, CXGate,PauliEvolutionGate
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.synthesis.evolution import SuzukiTrotter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator,QiskitRuntimeService
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions,SamplerOptions

## Measurement


In [77]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [78]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [79]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [80]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

In [81]:
def measure_ZX(qc, q0, q1, cbit):

    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

## Single qubit clifford group

In [82]:
def add_H(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)

    qc.x(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [83]:
def add_S(qc, d, a, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q)

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [84]:
def add_SH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZZ(qc, a, d, c[1])  # s1
    measure_ZY(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s0 s2 s3)/2} even parity
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)

    # Z^{(1 - s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)

    qc.reset(a)

    return qc

In [85]:
def add_HSH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    qc.reset(a)

    return qc

In [86]:
def add_HS(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZY(qc, a, d, c[1])  # s1
    measure_ZZ(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)

    qc.reset(a)

    return qc

In [87]:
# ZYZ Euler decomposation of 1-qubit su(2) with qiskit convention U3Gate

def add_U3(qc, d, theta, phi, lam):

    qc.rz(lam, d)

    qc.sdg(d)
    qc.h(d)
    qc.rz(theta, d)
    qc.h(d)
    qc.s(d)

    qc.rz(phi, d)

    return qc

## Single qubit gate in supremacy circuit

In [88]:
def add_sqrtX(qc, d, a, cbit):
    qc = add_HSH(qc, d, a, cbit)
    return qc

In [89]:
def add_sqrtY(qc, d, a, cbit):
   qc = add_S(qc, d, a, cbit)
   qc = add_HS(qc, d, a, cbit)
   return qc

In [90]:
def add_sqrtW(qc, d, a, cbit):
    qc.tdg(d)
    qc = add_sqrtX(qc, d, a, cbit)
    qc.t(d)
    return qc

## Two qubit gate in supremacy circuit

### fSim decomposition

The two-qubit gate fSim(θ, φ) can be decomposed into an optimal circuit using  
**3 CNOT gates and 8 single-qubit SU(2) gates**.

Each SU(2) can be expressed using the **ZYZ Euler decomposition**:

U3(θ, φ, λ) = RZ(λ) · RY(θ) · RZ(φ)

Solving all Euler angles analytically is difficult. For each fSim(θ, φ), we use Qiskit to compute these parameters numerically on a classical computer, and then feed them into our MBQC implementation.

---

### Circuit structure

$$
(U_0 \otimes U_1)\;\rightarrow\;\mathrm{CNOT}\;\rightarrow\;
(U_2 \otimes U_3)\;\rightarrow\;\mathrm{CNOT}\;\rightarrow\;
(U_4 \otimes U_5)\;\rightarrow\;\mathrm{CNOT}\;\rightarrow\;
(U_6 \otimes U_7)
$$

---

### Example

Below is an example of extracting these parameters using Qiskit:

In [91]:
# fSim matrix

def fSim(theta, phi):
    return np.array([
        [1, 0, 0, 0],
        [0, np.cos(theta), -1j*np.sin(theta), 0],
        [0, -1j*np.sin(theta), np.cos(theta), 0],
        [0, 0, 0, np.exp(-1j*phi)]
    ], dtype=complex)

In [92]:
# return A list of tuples: (qubit_index, θ, φ, λ), where qubit_index = 0 or 1

def get_u3_params(theta, phi):

    # Build circuit
    qc = QuantumCircuit(2)
    qc.append(UnitaryGate(fSim(theta, phi)), [0, 1])

    # 3-CNOT synthesis
    U_target = Operator(qc).data

    decomposer = TwoQubitBasisDecomposer(
        CXGate(),
        euler_basis="U3",
    )

    synth = decomposer(U_target)

    synth = transpile(
        synth,
        basis_gates=["u3", "cx", "u"],
        optimization_level=0,
    )

    # Flatten
    flat = synth.decompose(reps=10)

    # Extract U parameters
    u_list = []
    for instr in flat.data:
        inst = instr.operation
        qargs = instr.qubits

        if inst.name in ["u", "u3"]:
            theta_u, phi_u, lambda_u = inst.params
            qubit_index = flat.find_bit(qargs[0]).index
            u_list.append((qubit_index,
                           float(theta_u),
                           float(phi_u),
                           float(lambda_u)))

    return u_list

In [93]:
def add_CNOT(qc, c, a, t, cbit):
    # needs 8 cbits. all_ reuses cbit[0-4]. measure_ uses cbit[5-7]
    # Qubit A is initialized in an eigenstate of Z

    # prepare a in z-basis
    qc.reset(a)

    measure_ZX(qc, c, a, cbit[5])
    measure_ZX(qc, a, t, cbit[6])

    # this is single X-measurement on qubit a. h is on level of simulation, not circuit element
    qc.h(a)
    qc.measure(a, cbit[7])
    qc.h(a)

    # reset a to 0 for an easier comparision
    qc.reset(a)

    # we use cbit directly because it stores as 0(even) and 1(odd), as needed in paper
    P1 = cbit[5]
    P2 = cbit[6]
    M  = cbit[7]

    # X_t^{(P1 ⊕ M)}
    xt = expr.bit_xor(P1, M)
    with qc.if_test(xt):
        qc.x(t)

    #Z_c^{P2}
    with qc.if_test((P2, 1)):
        qc.z(c)

    return qc

In [94]:
def add_TwoQubitGate(qc, c, a, t, theta, phi, Z1, Z2, Z3, Z4, cbit):
    qc.rz(Z1, c)
    qc.rz(Z2, t)

    u_list = get_u3_params(theta, phi) # solve Euler angles

    # There are 4 layers of U3
    # layout: (U0,U1), CNOT, (U2,U3), CNOT, (U4,U5), CNOT, (U6,U7) acts on (c, t)

    for layer in range(4):
        # Apply the two U3 gates for this layer
        for i in range(2):

            idx = 2*layer + i
            qubit, theta, phi, lam = u_list[idx]

            # u_list only has qubit 0 and 1. Here we need c,a, t

            if qubit==0:
                qc = add_U3(qc, c, theta, phi, lam)
            else:
                qc = add_U3(qc, t, theta, phi, lam)

        # Add CNOT after first 3 layers
        if layer < 3:
            qc = add_CNOT(qc, c, a, t, cbit)   # fixed pattern

    qc.rz(Z3, c)
    qc.rz(Z4, t)

    return qc

## Example of fSim decomposation

In [95]:
theta = np.pi / 5
phi   = np.pi / 7

qc = QuantumCircuit(2)
qc.append(UnitaryGate(fSim(theta, phi)), [0, 1])


#3-CNOT synthesis
U_target = Operator(qc).data

decomposer = TwoQubitBasisDecomposer(
    CXGate(),
    euler_basis="U3",
)

synth = decomposer(U_target)

# clean into u3 + cx
synth = transpile(
    synth,
    basis_gates=["u3", "cx"],
    optimization_level=0,
)


print("\nSynthesized circuit:")
print(synth.draw("text"))

cx_count = synth.decompose(reps=20).count_ops().get("cx", 0)
print("\nCX count:", cx_count)

fid = process_fidelity(Operator(synth), Operator(qc))
print("Process fidelity:", fid)


Synthesized circuit:
global phase: 0.56973
       ┌──────────────────┐            ┌──────────────────┐          »
q_0: ──┤ U3(π/2,-π/2,π/4) ├────■───────┤ U3(π/5,-π/2,π/2) ├───────■──»
     ┌─┴──────────────────┴─┐┌─┴─┐┌────┴──────────────────┴────┐┌─┴─┐»
q_1: ┤ U3(1.7229,π/2,-3π/4) ├┤ X ├┤ U3(1.6787,-0.10741,0.7846) ├┤ X ├»
     └──────────────────────┘└───┘└────────────────────────────┘└───┘»
«         ┌─────────────────┐          ┌────────────────────┐
«q_0: ────┤ U3(π/14,-π,π/2) ├───────■──┤ U3(π/2,2.1318,π/2) ├
«     ┌───┴─────────────────┴────┐┌─┴─┐├───────────────────┬┘
«q_1: ┤ U3(1.4205,3.1184,-1.417) ├┤ X ├┤ U3(π/2,-2.7327,0) ├─
«     └──────────────────────────┘└───┘└───────────────────┘ 

CX count: 3
Process fidelity: 1.0


In [96]:
params = get_u3_params(theta, phi)
print("Euler angles (qubit, θ, φ, λ):")
for qb, th, ph, lam in params:
    print(f"q{qb}: θ={th:.6f}, φ={ph:.6f}, λ={lam:.6f}")

Euler angles (qubit, θ, φ, λ):
q0: θ=1.570796, φ=-1.570796, λ=0.785398
q1: θ=1.722868, φ=1.570796, λ=-2.356194
q0: θ=0.628319, φ=-1.570796, λ=1.570796
q1: θ=1.678657, φ=-0.107408, λ=0.784603
q0: θ=0.224399, φ=-3.141593, λ=1.570796
q1: θ=1.420493, φ=3.118381, λ=-1.416977
q0: θ=1.570796, φ=2.131795, λ=1.570796
q1: θ=1.570796, φ=-2.732665, λ=0.000000


# Test MBQC gates

For each gate, we test on 100 individual trials

For each trial:

- Prepare a **random state over the full Hilbert space**
- Use **random parameters** when applicable
- Execute both MBQC and GBQC circuits
- Compare the final states and check the overlap
$
\left| \langle \psi_{\mathrm{GBQC}} \mid \psi_{\mathrm{MBQC}} \rangle \right| = 1
$ up to tolerance 1e-6

---

This method ensures:

1. **Eliminates relative phase ambiguity**  

2. **Covers the full Hilbert space**  

3. **Samples measurement branches**  

4. **Validates arbitrary parameters**  

## Two-qubit gate test

In [97]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    psi02 = random_statevector(4)
    phi = random.uniform(0, 2*pi)
    theta = random.uniform(0, 2*pi)

    # -----------------------
    # DIRECT (your matrix)
    # -----------------------
    qc1 = QuantumCircuit(3)
    qc1.initialize(psi02.data, [0, 2])

    U2 = UnitaryGate(fSim(theta, phi))
    qc1.append(U2, [0, 2])

    sv1 = Statevector.from_instruction(qc1)

    # -----------------------
    # MBQC (your circuit)
    # -----------------------
    qc2 = QuantumCircuit(3)
    cbit2 = ClassicalRegister(8)
    qc2.add_register(cbit2)

    qc2.initialize(psi02.data, [0, 2])

    qc2 = add_TwoQubitGate(
        qc2,
        0,  # c
        1,  # a (ancilla)
        2,  # t
        theta,
        phi,
        0, 0, 0, 0,
        cbit2
    )

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    sv_mbqc = list(data.values())[0]

    # -----------------------
    # Compare
    # -----------------------
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All fSim tests pass (overlap ≈ 1)")

✅ All fSim tests pass (overlap ≈ 1)


## Single-qubit gate test

In [98]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    # --- random input state (1 qubit on data wire) ---
    psi = random_statevector(2)

    # --- gate-based (GBQC) ---
    qc1 = QuantumCircuit(2)
    qc1.initialize(psi.data, 0)
    qc1.sx(0)

    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC ---
    qc2 = QuantumCircuit(2)
    cbit2 = ClassicalRegister(5)
    qc2.add_register(cbit2)

    qc2.initialize(psi.data, 0)
    qc2 = add_sqrtX(qc2, 0, 1, cbit2)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract one branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    # check
    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All randomized sqrt(X) tests passed (overlap ≈ 1)")

✅ All randomized sqrt(X) tests passed (overlap ≈ 1)


In [99]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    # --- random input state (1 qubit on data wire) ---
    psi = random_statevector(2)

    # --- gate-based (GBQC) ---
    qc1 = QuantumCircuit(2)
    qc1.initialize(psi.data, 0)
    qc1.ry(np.pi/2, 0)

    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC ---
    qc2 = QuantumCircuit(2)
    cbit2 = ClassicalRegister(5)
    qc2.add_register(cbit2)

    qc2.initialize(psi.data, 0)
    qc2 = add_sqrtY(qc2, 0, 1, cbit2)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract one branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    # check
    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All randomized sqrt(Y) tests passed (overlap ≈ 1)")

✅ All randomized sqrt(Y) tests passed (overlap ≈ 1)


# Random circuit with single qubit gate

qubit geometry: $(i,0)=\text{data},\ (i,1)=\text{ancilla},\ (i,j)=2i+j$

Test with 4*2 qubits and 100 periods: random entangled state, random seed

In [100]:
def random_single_MBQC(qc, N, periods, seed):
    random.seed(seed)

    #all circuits with single-qubit clifford gates need at most 5 classical bits
    cbit = ClassicalRegister(8)

    qc.add_register(cbit)

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                qc = add_sqrtX(qc, 2*i, 2*i+1, cbit)
            elif gate == "sY":
                qc = add_sqrtY(qc, 2*i, 2*i+1, cbit)
            else:
                qc = add_sqrtW(qc, 2*i, 2*i+1, cbit)

            last_gate[i] = gate

    return qc

In [101]:
def random_single_gateQC(qc, N, periods, seed):
    random.seed(seed)

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                qc.sx(2*i)
            elif gate == "sY":
                qc.ry(np.pi/2, 2*i)
            else:
                qc.tdg(2*i)
                qc.sx(2*i)
                qc.t(2*i)

            last_gate[i] = gate

    return qc

In [102]:
sim = AerSimulator(method="statevector")

N = 4
periods = 100
num_trials = 10

print(f"single-qubit-gate circuit with 4*2 qubits and 100 periods")

for i in range(num_trials):

    psi = random_statevector(2**N)

    seed = np.random.randint(1000)

    # --- direct circuit ---
    qc1 = QuantumCircuit(2*N)
    qc1.initialize(psi.data, list(range(0, 2*N, 2)))
    qc1 = random_single_gateQC(qc1, N, periods, seed)
    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC circuit ---
    qc2 = QuantumCircuit(2*N)
    qc2.initialize(psi.data, list(range(0, 2*N, 2)))
    qc2 = random_single_MBQC(qc2, N, periods, seed)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))


    print(f"Trial {i+1}: overlap = {overlap}")

single-qubit-gate circuit with 4*2 qubits and 100 periods
Trial 1: overlap = 0.9999999999999871
Trial 2: overlap = 0.999999999999988
Trial 3: overlap = 0.9999999999999861
Trial 4: overlap = 0.9999999999999866
Trial 5: overlap = 0.9999999999999856
Trial 6: overlap = 0.9999999999999863
Trial 7: overlap = 0.9999999999999869
Trial 8: overlap = 0.9999999999999859
Trial 9: overlap = 0.9999999999999857
Trial 10: overlap = 0.9999999999999866
